In [1]:
import os
os.environ["HYDRA_FULL_ERROR"] = "1"
import logging

import hydra
import lightning as L


import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from lightning.pytorch.callbacks import (
    LearningRateMonitor,
    ModelCheckpoint,
    ModelSummary,
)
from omegaconf import DictConfig, OmegaConf
from temporaldata import Data

from torch_brain.registry import MODALITY_REGISTRY, ModalitySpec
from torch_brain.optim import SparseLamb
from torch_brain.models.poyo import POYO
from torch_brain.utils import callbacks as tbrain_callbacks
from torch_brain.utils import seed_everything
from torch_brain.utils.stitcher import (
    DecodingStitchEvaluator,
    DataForDecodingStitchEvaluator,
)
from torch_brain.data import Dataset, collate
from torch_brain.data.sampler import (
    DistributedStitchingFixedWindowSampler,
    RandomFixedWindowSampler,
    BalancedRandomFixedWindowSampler
)
from torch_brain.transforms import Compose

root = "D:/Pose/Neuro Code/data/NoveltySessInfoMatFiles/linear_processed/hippo_processed"

def get_dataset_config(brainset, sessions):
    brainset_norms = {
        "perich_miller_population_2018": {
            "mean": 0.0,
            "std": 20.0
        },
        "rat_hippocampus": {
            "mean": 0.0,
            "std": 1.0
        }
    }
    
    config = f"""
    - selection:
      - brainset: {brainset}
        sessions:"""
    if type(sessions) is not list:
        sessions = [sessions]
    for session in sessions:
        config += f"""
          - {session}"""
    rid = "cursor_velocity_2d" if brainset == "perich_miller_population_2018" else "linear_maze_pos"
    config += f"""
      config:
        readout:
          readout_id: {rid}
          normalize_mean: {brainset_norms[brainset]["mean"]}
          normalize_std: {brainset_norms[brainset]["std"]}
          metrics:
            - metric:
                _target_: torchmetrics.R2Score
    """

    config = OmegaConf.create(config)

    return config


d:\Anaconda\Lib\site-packages\transformers\utils\generic.py:441: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(


In [2]:
from torch.utils.data import DataLoader
from torch_brain.models.poyo import POYO
from torch_brain.registry import MODALITY_REGISTRY, ModalitySpec

cfg = get_dataset_config("rat_hippocampus",
        ["achilles_10252013_sessinfo",
         "buddy_06272013_sessinfo",
         "cicero_09012014_sessinfo", # 
         "gatsby_08022013_sessinfo"
         ])

readout_id = "linear_maze_pos"
readout_spec = MODALITY_REGISTRY[readout_id]
temp_model = POYO(sequence_length=1.0, readout_spec=readout_spec, latent_step=1/8.)

train_dataset = Dataset(
    root=root,
    config=cfg,
    split="train",
    transform=Compose([temp_model.tokenize]), # [?] read model tokenize
    # session_id_prefix_fn = lambda data: f"hippo1/",
    # unit_id_prefix_fn = lambda data: f"hippo1/",
    # subject_id_prefix_fn = lambda data: f"hippo1/",
)

temp_model.unit_emb.initialize_vocab(train_dataset.get_unit_ids())
temp_model.session_emb.initialize_vocab(train_dataset.get_session_ids())

# train_sampler = RandomFixedWindowSampler(
#     sampling_intervals=train_dataset.get_sampling_intervals(),
#     window_length=1.0,
# )
bs = 12

train_sampler = BalancedRandomFixedWindowSampler(
    sampling_intervals=train_dataset.get_sampling_intervals(),
    window_length=1.0,
    batch_size=bs, 
    subject_ids=train_dataset.get_session_ids()
)

train_loader = DataLoader(
    train_dataset,
    sampler=train_sampler,
    collate_fn=collate,
    batch_size=bs,
)

In [5]:
# print(train_dataset.get_sampling_intervals()['rat_hippocampus/achilles_10252013_sessinfo'].start, train_dataset.get_sampling_intervals()['rat_hippocampus/achilles_10252013_sessinfo'].end)
# print(train_dataset.get_sampling_intervals()['rat_hippocampus/buddy_06272013_sessinfo'].start, train_dataset.get_sampling_intervals()['rat_hippocampus/buddy_06272013_sessinfo'].end)

In [8]:
64 % 4 

4

In [6]:
for i, batch in enumerate(train_loader):
    # print(batch.keys()) # ['model_inputs', 'target_values', 'target_weights', 'session_id', 'absolute_start', 'eval_mask'] 
    # print('model_inputs', batch['model_inputs'].keys())

    for k, v in batch['model_inputs'].items():
        print(k, v.shape)


    # print('target_values', batch['target_values'].shape)
    # print('target_weights', batch['target_weights'].shape)
    # print('session_id', batch['session_id'])
    # print('absolute_start', batch['absolute_start'].shape)
    # print('eval_mask', batch['eval_mask'].shape)

    break

rat_hippocampus/achilles_10252013_sessinfo 158
rat_hippocampus/buddy_06272013_sessinfo 101
rat_hippocampus/cicero_09012014_sessinfo 728
rat_hippocampus/gatsby_08022013_sessinfo 261
input_unit_index torch.Size([12, 512])
input_timestamps torch.Size([12, 512])
input_token_type torch.Size([12, 512])
input_mask torch.Size([12, 512])
latent_index torch.Size([12, 512])
latent_timestamps torch.Size([12, 512])
output_session_index torch.Size([12, 40])
output_timestamps torch.Size([12, 40])
output_mask torch.Size([12, 40])


In [6]:
import random

def extend_dict_lists_randomly(data_dict):
    """
    Shuffle each list in the dictionary, then randomly extend them to match the max length.
    Elements are added with minimal repetition - cycling through all elements before repeating.
    
    Args:
        data_dict: Dictionary where values are lists
        
    Returns:
        Dictionary with all lists extended to the same length
    """
    if not data_dict:
        return {}
    
    # Shuffle each list in place
    for key in data_dict:
        random.shuffle(data_dict[key])
    
    # Find the maximum length across all lists
    max_len = max(len(lst) for lst in data_dict.values())
    
    # Extend each list to match max_len
    result_dict = {}
    for key, lst in data_dict.items():
        if len(lst) == max_len:
            result_dict[key] = lst[:]  # Store a copy
        else:
            num_to_add = max_len - len(lst)
            
            # Create a cycle of the shuffled list
            # If we need more elements than the list length, we'll cycle through
            # but shuffle again for each cycle to avoid predictable patterns
            extended = lst[:]
            remaining = num_to_add
            
            while remaining > 0:
                # Create a new shuffle of the original list for each cycle
                cycle = lst[:]
                random.shuffle(cycle)
                # Take only what we need from this cycle
                to_add = cycle[:remaining]
                extended.extend(to_add)
                remaining -= len(to_add)
            
            result_dict[key] = extended
    
    return result_dict

# Alternative approach using full cycles to be more explicit about repetition
def extend_dict_lists_randomly_v2(data_dict):
    """
    More explicit version that ensures minimal repetition.
    Creates full cycles of all elements, shuffling each cycle independently.
    """
    if not data_dict:
        return {}
    
    # Shuffle each list in place
    for key in data_dict:
        random.shuffle(data_dict[key])
    
    # Find the maximum length across all lists
    max_len = max(len(lst) for lst in data_dict.values())
    
    # Extend each list to match max_len
    result_dict = {}
    for key, lst in data_dict.items():
        if len(lst) == max_len:
            result_dict[key] = lst[:]
        else:
            num_to_add = max_len - len(lst)
            lst_len = len(lst)
            
            # Calculate full cycles and remainder
            full_cycles = num_to_add // lst_len
            remainder = num_to_add % lst_len
            
            extended = lst[:]
            
            # Add full cycles (each shuffled independently)
            for _ in range(full_cycles):
                cycle = lst[:]
                random.shuffle(cycle)
                extended.extend(cycle)
            
            # Add remaining elements (shuffled subset)
            if remainder > 0:
                cycle = lst[:]
                random.shuffle(cycle)
                extended.extend(cycle[:remainder])
            
            result_dict[key] = extended
    
    return result_dict


Original dictionary:
  group_a: [1, 2, 3] (length: 3)
  group_b: [4, 5, 6, 7, 8, 9, 10, 11, 12, 13] (length: 10)
  group_c: [14, 15, 16, 17] (length: 4)
  group_d: [18, 19] (length: 2)

After shuffling and random extension:
  group_a: [3, 2, 1, 3, 2, 1, 2, 1, 3, 1] (length: 10)
    Max repetitions of any element: 4, Min: 3
  group_b: [11, 13, 10, 4, 8, 5, 12, 6, 9, 7] (length: 10)
    Max repetitions of any element: 1, Min: 1
  group_c: [14, 17, 15, 16, 14, 17, 15, 16, 14, 15] (length: 10)
    Max repetitions of any element: 3, Min: 2
  group_d: [18, 19, 18, 19, 18, 19, 19, 18, 19, 18] (length: 10)
    Max repetitions of any element: 5, Min: 5

All lengths: [10, 10, 10, 10]
All same length? True

--- Testing specific case: 16 elements needed, list length 10 ---
Original list: [4, 2, 10, 5, 3, 6, 9, 1, 7, 8]
Full extended list: [4, 2, 10, 5, 3, 6, 9, 1, 7, 8, 5, 10, 1, 4, 8, 3, 2, 6, 9, 7, 8, 3, 7, 4, 6, 5]
Length: 26
Element counts:
  1: appears 2 time(s)
  2: appears 2 time(s)
  3: ap

In [10]:
batch_size = 64
K = batch_size // 4

starting_indices = torch.arange(0, batch_size, K)

for idx in torch.randperm(len(starting_indices)):
    start_idx = starting_indices[idx]
    end_idx = min(start_idx + K, batch_size)
    print(start_idx, end_idx)

tensor(48) tensor(64)
tensor(16) tensor(32)
tensor(0) tensor(16)
tensor(32) tensor(48)


In [ ]:
Logged metrics: dict_keys(['train_loss', 'rat_hippo/achilles_10252013_sessinfo', 'rat_hippo/buddy_06272013_sessinfo', 'rat_hippo/cicero_09012014_sessinfo', 'rat_hippo/gatsby_08022013_sessinfo', 'average_val_metric', 'epoch_time', 'model_weights/mean_unit_emb.weight', 'model_weights/std_unit_emb.weight', 'model_weights/mean_session_emb.weight', 'model_weights/std_session_emb.weight', 'model_weights/mean_token_type_emb.weight', 'model_weights/std_token_type_emb.weight', 'model_weights/mean_latent_emb.weight', 'model_weights/std_latent_emb.weight', 'model_weights/mean_enc_atn.norm.weight', 'model_weights/std_enc_atn.norm.weight', 'model_weights/mean_enc_atn.norm.bias', 'model_weights/std_enc_atn.norm.bias', 'model_weights/mean_enc_atn.norm_context.weight', 'model_weights/std_enc_atn.norm_context.weight', 'model_weights/mean_enc_atn.norm_context.bias', 'model_weights/std_enc_atn.norm_context.bias', 'model_weights/mean_enc_atn.to_q.weight', 'model_weights/std_enc_atn.to_q.weight', 'model_weights/mean_enc_atn.to_kv.weight', 'model_weights/std_enc_atn.to_kv.weight', 'model_weights/mean_enc_atn.to_out.weight', 'model_weights/std_enc_atn.to_out.weight', 'model_weights/mean_enc_atn.to_out.bias', 'model_weights/std_enc_atn.to_out.bias', 'model_weights/mean_enc_ffn.0.weight', 'model_weights/std_enc_ffn.0.weight', 'model_weights/mean_enc_ffn.0.bias', 'model_weights/std_enc_ffn.0.bias', 'model_weights/mean_enc_ffn.1.net.0.weight', 'model_weights/std_enc_ffn.1.net.0.weight', 'model_weights/mean_enc_ffn.1.net.0.bias', 'model_weights/std_enc_ffn.1.net.0.bias', 'model_weights/mean_enc_ffn.1.net.3.weight', 'model_weights/std_enc_ffn.1.net.3.weight', 'model_weights/mean_enc_ffn.1.net.3.bias', 'model_weights/std_enc_ffn.1.net.3.bias', 'model_weights/mean_proc_layers.0.0.norm.weight', 'model_weights/std_proc_layers.0.0.norm.weight', 'model_weights/mean_proc_layers.0.0.norm.bias', 'model_weights/std_proc_layers.0.0.norm.bias', 'model_weights/mean_proc_layers.0.0.to_qkv.weight', 'model_weights/std_proc_layers.0.0.to_qkv.weight', 'model_weights/mean_proc_layers.0.0.to_out.weight', 'model_weights/std_proc_layers.0.0.to_out.weight', 'model_weights/mean_proc_layers.0.0.to_out.bias', 'model_weights/std_proc_layers.0.0.to_out.bias', 'model_weights/mean_proc_layers.0.1.0.weight', 'model_weights/std_proc_layers.0.1.0.weight', 'model_weights/mean_proc_layers.0.1.0.bias', 'model_weights/std_proc_layers.0.1.0.bias', 'model_weights/mean_proc_layers.0.1.1.net.0.weight', 'model_weights/std_proc_layers.0.1.1.net.0.weight', 'model_weights/mean_proc_layers.0.1.1.net.0.bias', 'model_weights/std_proc_layers.0.1.1.net.0.bias', 'model_weights/mean_proc_layers.0.1.1.net.3.weight', 'model_weights/std_proc_layers.0.1.1.net.3.weight', 'model_weights/mean_proc_layers.0.1.1.net.3.bias', 'model_weights/std_proc_layers.0.1.1.net.3.bias', 'model_weights/mean_proc_layers.1.0.norm.weight', 'model_weights/std_proc_layers.1.0.norm.weight', 'model_weights/mean_proc_layers.1.0.norm.bias', 'model_weights/std_proc_layers.1.0.norm.bias', 'model_weights/mean_proc_layers.1.0.to_qkv.weight', 'model_weights/std_proc_layers.1.0.to_qkv.weight', 'model_weights/mean_proc_layers.1.0.to_out.weight', 'model_weights/std_proc_layers.1.0.to_out.weight', 'model_weights/mean_proc_layers.1.0.to_out.bias', 'model_weights/std_proc_layers.1.0.to_out.bias', 'model_weights/mean_proc_layers.1.1.0.weight', 'model_weights/std_proc_layers.1.1.0.weight', 'model_weights/mean_proc_layers.1.1.0.bias', 'model_weights/std_proc_layers.1.1.0.bias', 'model_weights/mean_proc_layers.1.1.1.net.0.weight', 'model_weights/std_proc_layers.1.1.1.net.0.weight', 'model_weights/mean_proc_layers.1.1.1.net.0.bias', 'model_weights/std_proc_layers.1.1.1.net.0.bias', 'model_weights/mean_proc_layers.1.1.1.net.3.weight', 'model_weights/std_proc_layers.1.1.1.net.3.weight', 'model_weights/mean_proc_layers.1.1.1.net.3.bias', 'model_weights/std_proc_layers.1.1.1.net.3.bias', 'model_weights/mean_proc_layers.2.0.norm.weight', 'model_weights/std_proc_layers.2.0.norm.weight', 'model_weights/mean_proc_layers.2.0.norm.bias', 'model_weights/std_proc_layers.2.0.norm.bias', 'model_weights/mean_proc_layers.2.0.to_qkv.weight', 'model_weights/std_proc_layers.2.0.to_qkv.weight', 'model_weights/mean_proc_layers.2.0.to_out.weight', 'model_weights/std_proc_layers.2.0.to_out.weight', 'model_weights/mean_proc_layers.2.0.to_out.bias', 'model_weights/std_proc_layers.2.0.to_out.bias', 'model_weights/mean_proc_layers.2.1.0.weight', 'model_weights/std_proc_layers.2.1.0.weight', 'model_weights/mean_proc_layers.2.1.0.bias', 'model_weights/std_proc_layers.2.1.0.bias', 'model_weights/mean_proc_layers.2.1.1.net.0.weight', 'model_weights/std_proc_layers.2.1.1.net.0.weight', 'model_weights/mean_proc_layers.2.1.1.net.0.bias', 'model_weights/std_proc_layers.2.1.1.net.0.bias', 'model_weights/mean_proc_layers.2.1.1.net.3.weight', 'model_weights/std_proc_layers.2.1.1.net.3.weight', 'model_weights/mean_proc_layers.2.1.1.net.3.bias', 'model_weights/std_proc_layers.2.1.1.net.3.bias', 'model_weights/mean_proc_layers.3.0.norm.weight', 'model_weights/std_proc_layers.3.0.norm.weight', 'model_weights/mean_proc_layers.3.0.norm.bias', 'model_weights/std_proc_layers.3.0.norm.bias', 'model_weights/mean_proc_layers.3.0.to_qkv.weight', 'model_weights/std_proc_layers.3.0.to_qkv.weight', 'model_weights/mean_proc_layers.3.0.to_out.weight', 'model_weights/std_proc_layers.3.0.to_out.weight', 'model_weights/mean_proc_layers.3.0.to_out.bias', 'model_weights/std_proc_layers.3.0.to_out.bias', 'model_weights/mean_proc_layers.3.1.0.weight', 'model_weights/std_proc_layers.3.1.0.weight', 'model_weights/mean_proc_layers.3.1.0.bias', 'model_weights/std_proc_layers.3.1.0.bias', 'model_weights/mean_proc_layers.3.1.1.net.0.weight', 'model_weights/std_proc_layers.3.1.1.net.0.weight', 'model_weights/mean_proc_layers.3.1.1.net.0.bias', 'model_weights/std_proc_layers.3.1.1.net.0.bias', 'model_weights/mean_proc_layers.3.1.1.net.3.weight', 'model_weights/std_proc_layers.3.1.1.net.3.weight', 'model_weights/mean_proc_layers.3.1.1.net.3.bias', 'model_weights/std_proc_layers.3.1.1.net.3.bias', 'model_weights/mean_proc_layers.4.0.norm.weight', 'model_weights/std_proc_layers.4.0.norm.weight', 'model_weights/mean_proc_layers.4.0.norm.bias', 'model_weights/std_proc_layers.4.0.norm.bias', 'model_weights/mean_proc_layers.4.0.to_qkv.weight', 'model_weights/std_proc_layers.4.0.to_qkv.weight', 'model_weights/mean_proc_layers.4.0.to_out.weight', 'model_weights/std_proc_layers.4.0.to_out.weight', 'model_weights/mean_proc_layers.4.0.to_out.bias', 'model_weights/std_proc_layers.4.0.to_out.bias', 'model_weights/mean_proc_layers.4.1.0.weight', 'model_weights/std_proc_layers.4.1.0.weight', 'model_weights/mean_proc_layers.4.1.0.bias', 'model_weights/std_proc_layers.4.1.0.bias', 'model_weights/mean_proc_layers.4.1.1.net.0.weight', 'model_weights/std_proc_layers.4.1.1.net.0.weight', 'model_weights/mean_proc_layers.4.1.1.net.0.bias', 'model_weights/std_proc_layers.4.1.1.net.0.bias', 'model_weights/mean_proc_layers.4.1.1.net.3.weight', 'model_weights/std_proc_layers.4.1.1.net.3.weight', 'model_weights/mean_proc_layers.4.1.1.net.3.bias', 'model_weights/std_proc_layers.4.1.1.net.3.bias', 'model_weights/mean_proc_layers.5.0.norm.weight', 'model_weights/std_proc_layers.5.0.norm.weight', 'model_weights/mean_proc_layers.5.0.norm.bias', 'model_weights/std_proc_layers.5.0.norm.bias', 'model_weights/mean_proc_layers.5.0.to_qkv.weight', 'model_weights/std_proc_layers.5.0.to_qkv.weight', 'model_weights/mean_proc_layers.5.0.to_out.weight', 'model_weights/std_proc_layers.5.0.to_out.weight', 'model_weights/mean_proc_layers.5.0.to_out.bias', 'model_weights/std_proc_layers.5.0.to_out.bias', 'model_weights/mean_proc_layers.5.1.0.weight', 'model_weights/std_proc_layers.5.1.0.weight', 'model_weights/mean_proc_layers.5.1.0.bias', 'model_weights/std_proc_layers.5.1.0.bias', 'model_weights/mean_proc_layers.5.1.1.net.0.weight', 'model_weights/std_proc_layers.5.1.1.net.0.weight', 'model_weights/mean_proc_layers.5.1.1.net.0.bias', 'model_weights/std_proc_layers.5.1.1.net.0.bias', 'model_weights/mean_proc_layers.5.1.1.net.3.weight', 'model_weights/std_proc_layers.5.1.1.net.3.weight', 'model_weights/mean_proc_layers.5.1.1.net.3.bias', 'model_weights/std_proc_layers.5.1.1.net.3.bias', 'model_weights/mean_dec_atn.norm.weight', 'model_weights/std_dec_atn.norm.weight', 'model_weights/mean_dec_atn.norm.bias', 'model_weights/std_dec_atn.norm.bias', 'model_weights/mean_dec_atn.norm_context.weight', 'model_weights/std_dec_atn.norm_context.weight', 'model_weights/mean_dec_atn.norm_context.bias', 'model_weights/std_dec_atn.norm_context.bias', 'model_weights/mean_dec_atn.to_q.weight', 'model_weights/std_dec_atn.to_q.weight', 'model_weights/mean_dec_atn.to_kv.weight', 'model_weights/std_dec_atn.to_kv.weight', 'model_weights/mean_dec_atn.to_out.weight', 'model_weights/std_dec_atn.to_out.weight', 'model_weights/mean_dec_atn.to_out.bias', 'model_weights/std_dec_atn.to_out.bias', 'model_weights/mean_dec_ffn.0.weight', 'model_weights/std_dec_ffn.0.weight', 'model_weights/mean_dec_ffn.0.bias', 'model_weights/std_dec_ffn.0.bias', 'model_weights/mean_dec_ffn.1.net.0.weight', 'model_weights/std_dec_ffn.1.net.0.weight', 'model_weights/mean_dec_ffn.1.net.0.bias', 'model_weights/std_dec_ffn.1.net.0.bias', 'model_weights/mean_dec_ffn.1.net.3.weight', 'model_weights/std_dec_ffn.1.net.3.weight', 'model_weights/mean_dec_ffn.1.net.3.bias', 'model_weights/std_dec_ffn.1.net.3.bias', 'model_weights/mean_readout.weight', 'model_weights/mean_readout.bias'])
Callback metrics: dict_keys(['lr-SparseLamb/pg1', 'lr-SparseLamb/pg2', 'train_loss', 'rat_hippo/achilles_10252013_sessinfo', 'rat_hippo/buddy_06272013_sessinfo', 'rat_hippo/cicero_09012014_sessinfo', 'rat_hippo/gatsby_08022013_sessinfo', 'average_val_metric', 'epoch_time', 'model_weights/mean_unit_emb.weight', 'model_weights/std_unit_emb.weight', 'model_weights/mean_session_emb.weight', 'model_weights/std_session_emb.weight', 'model_weights/mean_token_type_emb.weight', 'model_weights/std_token_type_emb.weight', 'model_weights/mean_latent_emb.weight', 'model_weights/std_latent_emb.weight', 'model_weights/mean_enc_atn.norm.weight', 'model_weights/std_enc_atn.norm.weight', 'model_weights/mean_enc_atn.norm.bias', 'model_weights/std_enc_atn.norm.bias', 'model_weights/mean_enc_atn.norm_context.weight', 'model_weights/std_enc_atn.norm_context.weight', 'model_weights/mean_enc_atn.norm_context.bias', 'model_weights/std_enc_atn.norm_context.bias', 'model_weights/mean_enc_atn.to_q.weight', 'model_weights/std_enc_atn.to_q.weight', 'model_weights/mean_enc_atn.to_kv.weight', 'model_weights/std_enc_atn.to_kv.weight', 'model_weights/mean_enc_atn.to_out.weight', 'model_weights/std_enc_atn.to_out.weight', 'model_weights/mean_enc_atn.to_out.bias', 'model_weights/std_enc_atn.to_out.bias', 'model_weights/mean_enc_ffn.0.weight', 'model_weights/std_enc_ffn.0.weight', 'model_weights/mean_enc_ffn.0.bias', 'model_weights/std_enc_ffn.0.bias', 'model_weights/mean_enc_ffn.1.net.0.weight', 'model_weights/std_enc_ffn.1.net.0.weight', 'model_weights/mean_enc_ffn.1.net.0.bias', 'model_weights/std_enc_ffn.1.net.0.bias', 'model_weights/mean_enc_ffn.1.net.3.weight', 'model_weights/std_enc_ffn.1.net.3.weight', 'model_weights/mean_enc_ffn.1.net.3.bias', 'model_weights/std_enc_ffn.1.net.3.bias', 'model_weights/mean_proc_layers.0.0.norm.weight', 'model_weights/std_proc_layers.0.0.norm.weight', 'model_weights/mean_proc_layers.0.0.norm.bias', 'model_weights/std_proc_layers.0.0.norm.bias', 'model_weights/mean_proc_layers.0.0.to_qkv.weight', 'model_weights/std_proc_layers.0.0.to_qkv.weight', 'model_weights/mean_proc_layers.0.0.to_out.weight', 'model_weights/std_proc_layers.0.0.to_out.weight', 'model_weights/mean_proc_layers.0.0.to_out.bias', 'model_weights/std_proc_layers.0.0.to_out.bias', 'model_weights/mean_proc_layers.0.1.0.weight', 'model_weights/std_proc_layers.0.1.0.weight', 'model_weights/mean_proc_layers.0.1.0.bias', 'model_weights/std_proc_layers.0.1.0.bias', 'model_weights/mean_proc_layers.0.1.1.net.0.weight', 'model_weights/std_proc_layers.0.1.1.net.0.weight', 'model_weights/mean_proc_layers.0.1.1.net.0.bias', 'model_weights/std_proc_layers.0.1.1.net.0.bias', 'model_weights/mean_proc_layers.0.1.1.net.3.weight', 'model_weights/std_proc_layers.0.1.1.net.3.weight', 'model_weights/mean_proc_layers.0.1.1.net.3.bias', 'model_weights/std_proc_layers.0.1.1.net.3.bias', 'model_weights/mean_proc_layers.1.0.norm.weight', 'model_weights/std_proc_layers.1.0.norm.weight', 'model_weights/mean_proc_layers.1.0.norm.bias', 'model_weights/std_proc_layers.1.0.norm.bias', 'model_weights/mean_proc_layers.1.0.to_qkv.weight', 'model_weights/std_proc_layers.1.0.to_qkv.weight', 'model_weights/mean_proc_layers.1.0.to_out.weight', 'model_weights/std_proc_layers.1.0.to_out.weight', 'model_weights/mean_proc_layers.1.0.to_out.bias', 'model_weights/std_proc_layers.1.0.to_out.bias', 'model_weights/mean_proc_layers.1.1.0.weight', 'model_weights/std_proc_layers.1.1.0.weight', 'model_weights/mean_proc_layers.1.1.0.bias', 'model_weights/std_proc_layers.1.1.0.bias', 'model_weights/mean_proc_layers.1.1.1.net.0.weight', 'model_weights/std_proc_layers.1.1.1.net.0.weight', 'model_weights/mean_proc_layers.1.1.1.net.0.bias', 'model_weights/std_proc_layers.1.1.1.net.0.bias', 'model_weights/mean_proc_layers.1.1.1.net.3.weight', 'model_weights/std_proc_layers.1.1.1.net.3.weight', 'model_weights/mean_proc_layers.1.1.1.net.3.bias', 'model_weights/std_proc_layers.1.1.1.net.3.bias', 'model_weights/mean_proc_layers.2.0.norm.weight', 'model_weights/std_proc_layers.2.0.norm.weight', 'model_weights/mean_proc_layers.2.0.norm.bias', 'model_weights/std_proc_layers.2.0.norm.bias', 'model_weights/mean_proc_layers.2.0.to_qkv.weight', 'model_weights/std_proc_layers.2.0.to_qkv.weight', 'model_weights/mean_proc_layers.2.0.to_out.weight', 'model_weights/std_proc_layers.2.0.to_out.weight', 'model_weights/mean_proc_layers.2.0.to_out.bias', 'model_weights/std_proc_layers.2.0.to_out.bias', 'model_weights/mean_proc_layers.2.1.0.weight', 'model_weights/std_proc_layers.2.1.0.weight', 'model_weights/mean_proc_layers.2.1.0.bias', 'model_weights/std_proc_layers.2.1.0.bias', 'model_weights/mean_proc_layers.2.1.1.net.0.weight', 'model_weights/std_proc_layers.2.1.1.net.0.weight', 'model_weights/mean_proc_layers.2.1.1.net.0.bias', 'model_weights/std_proc_layers.2.1.1.net.0.bias', 'model_weights/mean_proc_layers.2.1.1.net.3.weight', 'model_weights/std_proc_layers.2.1.1.net.3.weight', 'model_weights/mean_proc_layers.2.1.1.net.3.bias', 'model_weights/std_proc_layers.2.1.1.net.3.bias', 'model_weights/mean_proc_layers.3.0.norm.weight', 'model_weights/std_proc_layers.3.0.norm.weight', 'model_weights/mean_proc_layers.3.0.norm.bias', 'model_weights/std_proc_layers.3.0.norm.bias', 'model_weights/mean_proc_layers.3.0.to_qkv.weight', 'model_weights/std_proc_layers.3.0.to_qkv.weight', 'model_weights/mean_proc_layers.3.0.to_out.weight', 'model_weights/std_proc_layers.3.0.to_out.weight', 'model_weights/mean_proc_layers.3.0.to_out.bias', 'model_weights/std_proc_layers.3.0.to_out.bias', 'model_weights/mean_proc_layers.3.1.0.weight', 'model_weights/std_proc_layers.3.1.0.weight', 'model_weights/mean_proc_layers.3.1.0.bias', 'model_weights/std_proc_layers.3.1.0.bias', 'model_weights/mean_proc_layers.3.1.1.net.0.weight', 'model_weights/std_proc_layers.3.1.1.net.0.weight', 'model_weights/mean_proc_layers.3.1.1.net.0.bias', 'model_weights/std_proc_layers.3.1.1.net.0.bias', 'model_weights/mean_proc_layers.3.1.1.net.3.weight', 'model_weights/std_proc_layers.3.1.1.net.3.weight', 'model_weights/mean_proc_layers.3.1.1.net.3.bias', 'model_weights/std_proc_layers.3.1.1.net.3.bias', 'model_weights/mean_proc_layers.4.0.norm.weight', 'model_weights/std_proc_layers.4.0.norm.weight', 'model_weights/mean_proc_layers.4.0.norm.bias', 'model_weights/std_proc_layers.4.0.norm.bias', 'model_weights/mean_proc_layers.4.0.to_qkv.weight', 'model_weights/std_proc_layers.4.0.to_qkv.weight', 'model_weights/mean_proc_layers.4.0.to_out.weight', 'model_weights/std_proc_layers.4.0.to_out.weight', 'model_weights/mean_proc_layers.4.0.to_out.bias', 'model_weights/std_proc_layers.4.0.to_out.bias', 'model_weights/mean_proc_layers.4.1.0.weight', 'model_weights/std_proc_layers.4.1.0.weight', 'model_weights/mean_proc_layers.4.1.0.bias', 'model_weights/std_proc_layers.4.1.0.bias', 'model_weights/mean_proc_layers.4.1.1.net.0.weight', 'model_weights/std_proc_layers.4.1.1.net.0.weight', 'model_weights/mean_proc_layers.4.1.1.net.0.bias', 'model_weights/std_proc_layers.4.1.1.net.0.bias', 'model_weights/mean_proc_layers.4.1.1.net.3.weight', 'model_weights/std_proc_layers.4.1.1.net.3.weight', 'model_weights/mean_proc_layers.4.1.1.net.3.bias', 'model_weights/std_proc_layers.4.1.1.net.3.bias', 'model_weights/mean_proc_layers.5.0.norm.weight', 'model_weights/std_proc_layers.5.0.norm.weight', 'model_weights/mean_proc_layers.5.0.norm.bias', 'model_weights/std_proc_layers.5.0.norm.bias', 'model_weights/mean_proc_layers.5.0.to_qkv.weight', 'model_weights/std_proc_layers.5.0.to_qkv.weight', 'model_weights/mean_proc_layers.5.0.to_out.weight', 'model_weights/std_proc_layers.5.0.to_out.weight', 'model_weights/mean_proc_layers.5.0.to_out.bias', 'model_weights/std_proc_layers.5.0.to_out.bias', 'model_weights/mean_proc_layers.5.1.0.weight', 'model_weights/std_proc_layers.5.1.0.weight', 'model_weights/mean_proc_layers.5.1.0.bias', 'model_weights/std_proc_layers.5.1.0.bias', 'model_weights/mean_proc_layers.5.1.1.net.0.weight', 'model_weights/std_proc_layers.5.1.1.net.0.weight', 'model_weights/mean_proc_layers.5.1.1.net.0.bias', 'model_weights/std_proc_layers.5.1.1.net.0.bias', 'model_weights/mean_proc_layers.5.1.1.net.3.weight', 'model_weights/std_proc_layers.5.1.1.net.3.weight', 'model_weights/mean_proc_layers.5.1.1.net.3.bias', 'model_weights/std_proc_layers.5.1.1.net.3.bias', 'model_weights/mean_dec_atn.norm.weight', 'model_weights/std_dec_atn.norm.weight', 'model_weights/mean_dec_atn.norm.bias', 'model_weights/std_dec_atn.norm.bias', 'model_weights/mean_dec_atn.norm_context.weight', 'model_weights/std_dec_atn.norm_context.weight', 'model_weights/mean_dec_atn.norm_context.bias', 'model_weights/std_dec_atn.norm_context.bias', 'model_weights/mean_dec_atn.to_q.weight', 'model_weights/std_dec_atn.to_q.weight', 'model_weights/mean_dec_atn.to_kv.weight', 'model_weights/std_dec_atn.to_kv.weight', 'model_weights/mean_dec_atn.to_out.weight', 'model_weights/std_dec_atn.to_out.weight', 'model_weights/mean_dec_atn.to_out.bias', 'model_weights/std_dec_atn.to_out.bias', 'model_weights/mean_dec_ffn.0.weight', 'model_weights/std_dec_ffn.0.weight', 'model_weights/mean_dec_ffn.0.bias', 'model_weights/std_dec_ffn.0.bias', 'model_weights/mean_dec_ffn.1.net.0.weight', 'model_weights/std_dec_ffn.1.net.0.weight', 'model_weights/mean_dec_ffn.1.net.0.bias', 'model_weights/std_dec_ffn.1.net.0.bias', 'model_weights/mean_dec_ffn.1.net.3.weight', 'model_weights/std_dec_ffn.1.net.3.weight', 'model_weights/mean_dec_ffn.1.net.3.bias', 'model_weights/std_dec_ffn.1.net.3.bias', 'model_weights/mean_readout.weight', 'model_weights/mean_readout.bias'])